# Sauti — Data Exploration

Use this notebook to:
- Explore collected raw data
- Check language distribution
- Preview messages before annotation
- Run the baseline model on samples
- Track annotation progress

In [30]:
import sys
import pandas as pd
import json
from pathlib import Path

# Add project root to path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

print(f'Project root: {ROOT}')
print(f'Python: {sys.version}')

Project root: /mnt/d/DS PROJECTS/sauti
Python: 3.11.15 (main, Mar 11 2026, 17:20:07) [GCC 14.3.0]


## 1. Load collected data

In [31]:
# Load the latest annotation CSV
import glob

csv_files = sorted(glob.glob(str(ROOT / 'data/processed/for_annotation_*.csv')))

if not csv_files:
    print('No annotation CSV found. Run the Telegram collector first.')
else:
    latest = csv_files[-1]
    print(f'Loading: {latest}')
    df = pd.read_csv(latest)
    print(f'Total records: {len(df)}')
    df.head()

Loading: /mnt/d/DS PROJECTS/sauti/data/processed/for_annotation_20260320.csv
Total records: 300


## 2. Source breakdown

In [32]:
# print('Records per source:')
# print(df['source'].value_counts())
# print()
# print('Text length stats:')
# df['length'] = df['text'].str.len()
# df['length'].describe().round(0)


# import glob
# import pandas as pd

# csv_files = sorted(glob.glob(str(ROOT / 'data/processed/for_annotation_*.csv')))
# print('Files found:')
# for f in csv_files:
#     print(f'  {f}')

# # Load and merge all
# dfs = [pd.read_csv(f) for f in csv_files]
# df = pd.concat(dfs, ignore_index=True)

# # Deduplicate by text
# df = df.drop_duplicates(subset='text')
# df['length'] = df['text'].str.len()

# print(f'\nTotal unique records: {len(df)}')
# print('\nRecords per source:')
# print(df['source'].value_counts())

# Replace cell 2 with this
import json
import pandas as pd
from pathlib import Path

ROOT = Path().resolve().parent

# Load all JSONL files directly
raw_files = sorted((ROOT / 'data' / 'raw').glob('telegram_*.jsonl'))
print('JSONL files found:')
for f in raw_files:
    lines = sum(1 for _ in open(f))
    print(f'  {f.name} — {lines} records')

# Merge all
records = []
for f in raw_files:
    with open(f, encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if line:
                records.append(json.loads(line))

df = pd.DataFrame(records)
df = df.drop_duplicates(subset='text')
df['length'] = df['text'].str.len()

print(f'\nTotal unique records: {len(df)}')
print('\nRecords per source:')
print(df['source'].value_counts())

JSONL files found:
  telegram_20260320_152015.jsonl — 723 records
  telegram_20260320_152127.jsonl — 723 records
  telegram_20260320_154832.jsonl — 1589 records
  telegram_20260320_155155.jsonl — 300 records

Total unique records: 2055

Records per source:
source
telegram_thecivican           300
telegram_tuko_news            300
telegram_citizentvke          299
telegram_DailyNation          295
telegram_Kenyanyouthlinks     239
telegram_Nairobby             239
telegram_confessionsbytunu    228
telegram_thestarkenya         116
telegram_TUKOCommunity         24
telegram_DailyNationNews        8
telegram_ntvkenyaofficial       6
telegram_KTNNewskenya           1
Name: count, dtype: int64


## 3. Language detection

In [33]:
# Detect language of each message
try:
    from langdetect import detect, LangDetectException

    def detect_lang(text):
        try:
            return detect(str(text))
        except LangDetectException:
            return 'unknown'

    print('Detecting languages...')
    df['detected_language'] = df['text'].apply(detect_lang)
    print(df['detected_language'].value_counts().head(10))

except ImportError:
    print('langdetect not installed. Run: pip install langdetect')

Detecting languages...


detected_language
en    1861
sw      86
id      21
de      11
fr      10
tl      10
it       8
nl       7
ca       7
et       5
Name: count, dtype: int64


## 4. Browse messages by source

In [34]:
# Change source name to browse different channels
source = 'telegram_DailyNation'

sample = df[df['source'] == source]['text'].head(20).tolist()
for i, text in enumerate(sample, 1):
    print(f'[{i}] {text[:200]}')
    print()

[1] At the centre of this “Uko kadi?” growing movement is 26-year-old Allans Ademba, who was born and raised in Kibra, Nairobi. A photojournalist by profession, he says, his work has focused on helping pe

[2] James Kafani Mzungu, a land agent, was escorting businessman Sidik Anverali Sumra to Junju village in Kilifi County to view a 90-acre piece of land on the morning of July 7, 2021.

They came for land.

[3] Three presidents, one promise: Ruto faces Mau Forest reality test ahead of 2027

[4] Meet Allans Ademba, man behind ‘Uko Kadi?’ voter registration trend

[5] Murder Tapes: How probe failures, missing evidence sank Kilifi triple killing case

[6] Chad plans to send 800 security personnel to Haiti, official says

[7] 'I quit!': Mauritius deputy PM resigns after boss fails to appoint Finance minister

[8] Kindiki invites China VP Han to Kenya

[9] One woman's near-death story exposes Kenya's medical infrastructure crisis
#HealthyNation

[10] There are several beliefs about how a c

## 5. Run baseline model on sample

In [35]:
import pickle

# Load trained model
model_files = sorted((ROOT / 'ml' / 'runs').glob('*.pkl'))

if not model_files:
    print('No trained model found. Run training first.')
else:
    model_path = model_files[-1]
    print(f'Loading model: {model_path.name}')

    with open(model_path, 'rb') as f:
        model = pickle.load(f)

    from ml.src.data.cleaner import TextCleaner
    cleaner = TextCleaner()

    print('Model loaded successfully.')

Loading model: baseline_seed_v1.pkl
Model loaded successfully.


In [36]:
# Run predictions on a sample of collected messages
sample_texts = df['text'].dropna().sample(20, random_state=42).tolist()

results = []
for text in sample_texts:
    cleaned = cleaner.clean(text)['cleaned']
    pred = model.predict_single(cleaned, threshold=0.4)
    top = pred['predictions'][0] if pred['predictions'] else {}
    results.append({
        'text': text[:100],
        'label': top.get('label', 'none'),
        'confidence': top.get('confidence', 0),
    })

results_df = pd.DataFrame(results)
results_df

,text,label,confidence
0,Mudavadi: Even though the Judiciary must be in...,manipulation,0.4509
1,Eric Omondi Leads Nairobi Residents In A Massi...,manipulation,0.4601
2,Raila Odinga visits Nairobi CBD ahead of Wedne...,offensive_language,0.4251
3,Kenyan father heartbroken as Australia-based s...,manipulation,0.4938
4,Join the yellow card waitlist if you haven't,offensive_language,0.4724
5,**Authorities hunt man who distributes weed co...,gaslighting,0.4380
6,Drama As Woman Complains About Her Husband’s H...,manipulation,0.4453
7,"Another Woman records herself tearing up a 1,0...",manipulation,0.4403
8,North Rift leaders call out Raila over maandam...,manipulation,0.4865
9,Wangapi walipata hii ya mobiworkx jana ? 💦,hate_speech,0.4474


In [37]:
# Label distribution across predictions
print('Predicted label distribution:')
print(results_df['label'].value_counts())

Predicted label distribution:
label
manipulation          12
offensive_language     4
gaslighting            3
hate_speech            1
Name: count, dtype: int64


## 6. Find potentially harmful content to prioritize for annotation

In [38]:
# Run model on ALL collected messages and flag non-clean ones
# These are the ones most worth annotating first

print('Scanning all messages... (this may take a moment)')

flagged = []
for _, row in df.iterrows():
    cleaned = cleaner.clean(str(row['text']))['cleaned']
    pred = model.predict_single(cleaned, threshold=0.5)
    top = pred['predictions'][0] if pred['predictions'] else {}
    label = top.get('label', 'clean')
    conf = top.get('confidence', 0)

    if label != 'clean' and conf >= 0.5:
        flagged.append({
            'text': row['text'][:150],
            'source': row['source'],
            'label': label,
            'confidence': round(conf, 3),
        })

flagged_df = pd.DataFrame(flagged).sort_values('confidence', ascending=False)
print(f'Flagged {len(flagged_df)} messages as potentially harmful')
print()
print('Label breakdown:')
print(flagged_df['label'].value_counts())
flagged_df.head(20)

Scanning all messages... (this may take a moment)
Flagged 85 messages as potentially harmful

Label breakdown:
label
manipulation          60
offensive_language    16
gaslighting            6
hate_speech            3
Name: count, dtype: int64


,text,source,label,confidence
47,If you have this fanya trade ya convert 10$,telegram_Kenyanyouthlinks,manipulation,0.581
77,"""Uwachane na mimi nitengeneze Meru, Mt Kenya, ...",telegram_citizentvke,offensive_language,0.564
45,Do not fade any project you never know,telegram_Kenyanyouthlinks,offensive_language,0.547
0,"For sale ,plz give me message,",telegram_KTNNewskenya,manipulation,0.545
7,Parliament staff sacked over fake certificates...,telegram_DailyNation,manipulation,0.543
71,😂we pea huyo kijana mechi..\n\nMeanwhile conti...,telegram_confessionsbytunu,manipulation,0.542
74,Hawa ni wale hawakuli fare😂😂..,telegram_confessionsbytunu,hate_speech,0.540
70,You are sharp if you get this right😁,telegram_confessionsbytunu,offensive_language,0.540
67,The love of my life blocked me after flying ab...,telegram_confessionsbytunu,manipulation,0.530
62,After having sex for 5 years unakuja kutu uliz...,telegram_confessionsbytunu,gaslighting,0.528


## 7. Export flagged messages for priority annotation

In [39]:
# Save flagged messages to CSV for manual review
out = ROOT / 'data' / 'processed' / 'priority_annotation.csv'
flagged_df.to_csv(out, index=False)
print(f'Saved {len(flagged_df)} flagged messages → {out}')

Saved 85 flagged messages → /mnt/d/DS PROJECTS/sauti/data/processed/priority_annotation.csv


## 8. Raw JSONL explorer

In [40]:
# Browse raw collected JSONL files
raw_files = sorted((ROOT / 'data' / 'raw').glob('*.jsonl'))
print('Raw files collected:')
for f in raw_files:
    lines = sum(1 for _ in open(f))
    print(f'  {f.name} — {lines} records')

Raw files collected:
  telegram_20260320_152015.jsonl — 723 records
  telegram_20260320_152127.jsonl — 723 records


  telegram_20260320_154832.jsonl — 1589 records
  telegram_20260320_155155.jsonl — 300 records


In [41]:
# Preview a raw file
if raw_files:
    with open(raw_files[-1]) as f:
        sample = [json.loads(l) for l in f.readlines()[:5]]

    for r in sample:
        print(f"Source:  {r['source']}")
        print(f"Channel: {r.get('channel', 'n/a')}")
        print(f"Text:    {r['text'][:200]}")
        print()

Source:  telegram_citizentvke
Channel: Citizen Digital
Text:    Catholic bishops condemn President Ruto, opposition over ‘verbal indiscipline’ 
Catholic Bishops have criticised the country’s leadership over what it describes as rising cases of “verbal indiscipline

Source:  telegram_citizentvke
Channel: Citizen Digital
Text:    Public universities drown in over Ksh.100 billion debt, MPs demand answers

Source:  telegram_citizentvke
Channel: Citizen Digital
Text:    US–Iran conflict disrupts shipping as pressure mounts at Mombasa, Lamu ports

Source:  telegram_citizentvke
Channel: Citizen Digital
Text:    Gov Mutula Jnr bets on MKJ Supa Cup to bridge unemployment gap

Source:  telegram_citizentvke
Channel: Citizen Digital
Text:    World Bank bans PwC affiliates in Mauritius, Kenya, and Rwanda for 21 months after admitting to collusive and fraudulent practices involving a power integration project in Ethiopia



In [42]:
out = ROOT / 'data' / 'processed' / 'priority_annotation.csv'
flagged_df.to_csv(out, index=False)
print(f'Saved {len(flagged_df)} priority messages → {out}')
print()
print('Top channels in flagged set:')
print(flagged_df['source'].value_counts())

Saved 85 priority messages → /mnt/d/DS PROJECTS/sauti/data/processed/priority_annotation.csv

Top channels in flagged set:
source
telegram_Kenyanyouthlinks     21
telegram_confessionsbytunu    17
telegram_tuko_news            15
telegram_citizentvke           8
telegram_DailyNation           8
telegram_Nairobby              8
telegram_thecivican            5
telegram_KTNNewskenya          1
telegram_TUKOCommunity         1
telegram_DailyNationNews       1
Name: count, dtype: int64


In [44]:
# Add this as a new cell in your notebook
from facebook_scraper import get_posts

posts = []
pages_to_scrape = ["NairobiGossip", "KOT254", "KenyanMemes"]

for page in pages_to_scrape:
    print(f"Scraping {page}...")
    try:
        count = 0
        for post in get_posts(page, pages=5):
            if post.get('text'):
                posts.append({
                    'text': post['text'],
                    'source': f'facebook_{page}',
                    'likes': post.get('likes', 0),
                    'comments': post.get('comments', 0),
                })
                count += 1
        print(f"  Got {count} posts from {page}")
    except Exception as e:
        print(f"  Failed {page}: {e}")

print(f"\nTotal: {len(posts)} posts")

Scraping NairobiGossip...
  Got 0 posts from NairobiGossip
Scraping KOT254...
  Got 0 posts from KOT254
Scraping KenyanMemes...
  Got 0 posts from KenyanMemes

Total: 0 posts


In [1]:
import json
import pandas as pd
from pathlib import Path

ROOT = Path().resolve().parent

# Load all raw JSONL
raw_files = sorted((ROOT / 'data' / 'raw').glob('telegram_*.jsonl'))
records = []
for f in raw_files:
    with open(f, encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if line:
                records.append(json.loads(line))

df = pd.DataFrame(records).drop_duplicates(subset='text')

# Filter for channels most likely to have target content
priority_channels = [
    'telegram_confessionsbytunu',  # gaslighting, manipulation
    'telegram_thecivican',         # distress_trigger, political threats
    'telegram_Nairobby',           # offensive, manipulation
    'telegram_TUKOCommunity',      # community discussion
]

targeted = df[df['source'].isin(priority_channels)][['id','text','language','source']]
targeted = targeted[targeted['text'].str.len() > 30]

# Save for Label Studio import
out = ROOT / 'data' / 'processed' / 'annotation_round2.csv'
targeted.to_csv(out, index=False)
print(f'Exported {len(targeted)} messages for annotation round 2')
print(targeted['source'].value_counts())

Exported 723 messages for annotation round 2
source
telegram_thecivican           296
telegram_Nairobby             231
telegram_confessionsbytunu    182
telegram_TUKOCommunity         14
Name: count, dtype: int64
